# Pashto FastSpeech2: align all clips (Colab, before training on Kaggle)

Trains a Montreal Forced Aligner model on a sample of the clips, aligns **all** clips with it, and gives you
`pashto_aligned.zip` (the TextGrids + the aligner model, a few MB) to upload to Kaggle as a dataset for
`kaggle_train.ipynb`.

**Runtime: CPU** (*Runtime -> Change runtime type -> CPU*). The aligner doesn't use a GPU, so a GPU runtime only
spends your GPU quota. Keep the tab open while it runs; the result is also copied to Google Drive in case the
browser download fails.

## 1. Settings
`SAMPLE` clips (spread over all recordings) train the aligner: 200 clips is ~1.7 h of speech, enough
for one speaker and much faster than all 759. `None` trains on all clips (several times slower).

In [ ]:
SAMPLE    = 200                                  # clips to train the aligner on; None = all 759
DRIVE_DIR = "/content/drive/MyDrive/pashto_tts"  # a copy of the result goes here; None = no Drive
import os
JOBS = os.cpu_count()
print("CPU cores:", JOBS)

## 2. Get the repo and install (about 5 min, mostly the aligner)

In [ ]:
if not os.path.exists("/content/pashto-fastspeech2"):
    !git clone -q https://github.com/msds25033/pashto-fastspeech2.git /content/pashto-fastspeech2
%cd /content/pashto-fastspeech2
!git pull -q && git log -1 --format="%h %s"
!pip install -q -r requirements.txt
!python pipeline.py setup
!python pipeline.py install-mfa

## 3. Train the aligner on the sample
Saved to `runs/full/pashto_acoustic.zip`.

In [ ]:
SAMPLE_ARG = f"--sample {SAMPLE}" if SAMPLE else ""
!python pipeline.py corpus {SAMPLE_ARG}
!python pipeline.py align --train-acoustic --run-name full --jobs {JOBS}

## 4. Align all clips with it

In [ ]:
!python pipeline.py corpus
!python pipeline.py align --acoustic-model runs/full/pashto_acoustic.zip --jobs {JOBS}

## 5. Zip, copy to Drive, download
The zip keeps the folder layout `kaggle_train.py` looks for: `preprocessed_data/Pashto/TextGrid/pashto/*.TextGrid`
and `runs/full/pashto_acoustic.zip`.

In [ ]:
import glob, shutil, zipfile
grids = sorted(glob.glob("FastSpeech2/preprocessed_data/Pashto/TextGrid/pashto/*.TextGrid"))
print(len(grids), "TextGrids for", len(glob.glob("FastSpeech2/raw_data/Pashto/pashto/*.lab")), "clips")
with zipfile.ZipFile("pashto_aligned.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for g in grids:
        z.write(g, g[len("FastSpeech2/"):])
    z.write("runs/full/pashto_acoustic.zip", "runs/full/pashto_acoustic.zip")
print("pashto_aligned.zip:", round(os.path.getsize("pashto_aligned.zip") / 1e6, 1), "MB")
if DRIVE_DIR:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    shutil.copy("pashto_aligned.zip", DRIVE_DIR)
    print("copied to", DRIVE_DIR)
from google.colab import files
files.download("pashto_aligned.zip")

## 6. Upload to Kaggle
kaggle.com -> **Create -> New Dataset** -> upload `pashto_aligned.zip` (Kaggle unpacks it) -> name it e.g.
`pashto-aligned` -> Create. Then in `kaggle_train.ipynb`: **Add Input -> Your Work -> Datasets -> pashto-aligned**.